In [2]:
import torch
from dataclasses import dataclass
from typing import Optional, Tuple

@dataclass
class StreamResamplerState:
    B: int
    L: int
    out_len: int
    left: torch.Tensor      # [out_len] int64
    right: torch.Tensor     # [out_len] int64
    w: torch.Tensor         # [out_len] float32
    buf: dict               # idx -> tensor [B]
    i_in: int               # next input index expected
    j_out: int              # next output index to produce
    cycles: int             # total cycles elapsed

class LinResample1DStreamer:
    """
    Streaming simulator for 1D linear resampling.
    - One input sample per cycle: push via step(x_i) where x_i is [B] FP32.
    - Each cycle produces at most one output y_j ([B]) if required samples exist; else stalls.

    mode:
      - 'linspace': exact same src_pos as torch.linspace (matches your reference exactly)
      - 'iter'    : online pos += step in float32 (may differ by tiny eps)
    """
    def __init__(self, L: int, out_len: int, B: int = 1, device=None, mode: str = "linspace"):
        assert mode in ("linspace", "iter")
        self.device = device if device is not None else torch.device("cpu")
        self.mode = mode
        self.reset(L=L, out_len=out_len, B=B)

    def reset(self, L: int, out_len: int, B: int = 1):
        self.L = int(L)
        self.out_len = int(out_len)
        self.B = int(B)

        # src_pos generation
        if self.out_len == 1:
            src_pos = torch.zeros((1,), dtype=torch.float32, device=self.device)
        else:
            if self.mode == "linspace":
                src_pos = torch.linspace(0, self.L - 1, steps=self.out_len,
                                         dtype=torch.float32, device=self.device)
            else:
                # iterative float32 accumulation
                step = torch.tensor((self.L - 1) / (self.out_len - 1),
                                    dtype=torch.float32, device=self.device)
                src_pos = torch.empty((self.out_len,), dtype=torch.float32, device=self.device)
                pos = torch.tensor(0.0, dtype=torch.float32, device=self.device)
                for j in range(self.out_len):
                    src_pos[j] = pos
                    pos = pos + step
                src_pos[-1] = torch.tensor(float(self.L - 1), dtype=torch.float32, device=self.device)

        left = torch.floor(src_pos).to(torch.int64).clamp(0, self.L - 1)
        right = (left + 1).clamp(0, self.L - 1)
        w = (src_pos - left.to(torch.float32)).to(torch.float32)  # [out_len]

        self.state = StreamResamplerState(
            B=self.B, L=self.L, out_len=self.out_len,
            left=left, right=right, w=w,
            buf={}, i_in=0, j_out=0, cycles=0
        )

    @property
    def done(self) -> bool:
        return self.state.j_out >= self.state.out_len

    def step(self, x_i: Optional[torch.Tensor] = None) -> Tuple[bool, Optional[torch.Tensor], dict]:
        """
        Advance one cycle.
        Args:
            x_i: [B] FP32, next input sample. If None, no input this cycle.
        Returns:
            valid: whether output produced this cycle
            y: [B] if valid else None
            info: debug dict
        """
        st = self.state

        # push input (one per cycle)
        if x_i is not None:
            assert x_i.shape == (st.B,), f"x_i must be shape [B]={st.B}, got {tuple(x_i.shape)}"
            assert x_i.dtype == torch.float32, "x_i must be FP32"
            assert st.i_in < st.L, "All inputs already pushed"
            st.buf[st.i_in] = x_i.to(self.device).clone()
            st.i_in += 1

        y = None
        valid = False

        # try produce one output
        if st.j_out < st.out_len:
            l = int(st.left[st.j_out].item())
            r = int(st.right[st.j_out].item())
            if (l in st.buf) and (r in st.buf):
                wl = st.w[st.j_out]
                y = (1.0 - wl) * st.buf[l] + wl * st.buf[r]
                valid = True
                st.j_out += 1

                # optional buffer GC (monotonic mapping)
                min_keep = max(0, l - 2)
                for k in list(st.buf.keys()):
                    if k < min_keep:
                        del st.buf[k]

        st.cycles += 1
        info = {
            "cycle": st.cycles,
            "i_in": st.i_in,
            "j_out": st.j_out,
            "next_left": int(st.left[st.j_out].item()) if st.j_out < st.out_len else None,
            "next_right": int(st.right[st.j_out].item()) if st.j_out < st.out_len else None,
            "buf_size": len(st.buf),
        }
        return valid, y, info

# --- Your reference implementation (for comparison) ---
def lin_resample_1d_ref(x: torch.Tensor, out_len: int) -> torch.Tensor:
    if x.ndim == 1:
        x = x[None, :]
    B, L = x.shape
    if L == out_len:
        return x
    src_pos = torch.linspace(0, L - 1, steps=out_len, device=x.device, dtype=x.dtype)
    left = torch.floor(src_pos).long().clamp(0, L - 1)
    right = (left + 1).clamp(0, L - 1)
    w = (src_pos - left.to(x.dtype))[None, :]
    x_left = x[:, left]
    x_right = x[:, right]
    y = (1 - w) * x_left + w * x_right
    return y

def run_streaming_full(x: torch.Tensor, out_len: int, mode: str = "linspace") -> Tuple[torch.Tensor, int]:
    assert x.dtype == torch.float32
    if x.ndim == 1:
        x = x[None, :]
    B, L = x.shape

    streamer = LinResample1DStreamer(L=L, out_len=out_len, B=B, device=x.device, mode=mode)

    ys = []
    i = 0
    cycles = 0
    while not streamer.done:
        x_i = x[:, i] if i < L else None
        if i < L:
            i += 1
        valid, y, info = streamer.step(x_i)
        if valid:
            ys.append(y)
        cycles += 1
        if cycles > (L + out_len + 10000):
            raise RuntimeError("Too many cycles; check logic.")
    y_out = torch.stack(ys, dim=1)  # [B,out_len]
    return y_out, cycles

# --- Quick test ---
if __name__ == "__main__":
    torch.manual_seed(0)
    B, L, out_len = 1, 32, 64
    x = torch.randn(B, L, dtype=torch.float32)

    y_ref = lin_resample_1d_ref(x, out_len)
    y_stream, cyc = run_streaming_full(x, out_len, mode="linspace")

    print("cycles:", cyc)
    print("max abs diff:", (y_ref - y_stream).abs().max().item())
    print("mean abs diff:", (y_ref - y_stream).abs().mean().item())

cycles: 65
max abs diff: 0.0
mean abs diff: 0.0


In [3]:
import torch
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Any

@dataclass
class StreamResamplerState:
    B: int
    L: int
    out_len: int
    left: torch.Tensor      # [out_len] int64
    right: torch.Tensor     # [out_len] int64
    w: torch.Tensor         # [out_len] float32
    buf: dict               # idx -> tensor [B]
    i_in: int               # next input index expected
    j_out: int              # next output index to produce
    cycles: int             # total cycles elapsed

class LinResample1DStreamer:
    def __init__(self, L: int, out_len: int, B: int = 1, device=None, mode: str = "linspace"):
        assert mode in ("linspace", "iter")
        self.device = device if device is not None else torch.device("cpu")
        self.mode = mode
        self.reset(L=L, out_len=out_len, B=B)

    def reset(self, L: int, out_len: int, B: int = 1):
        self.L = int(L)
        self.out_len = int(out_len)
        self.B = int(B)

        # src_pos generation
        if self.out_len == 1:
            src_pos = torch.zeros((1,), dtype=torch.float32, device=self.device)
        else:
            if self.mode == "linspace":
                src_pos = torch.linspace(0, self.L - 1, steps=self.out_len,
                                         dtype=torch.float32, device=self.device)
            else:
                # iterative float32 accumulation
                step = torch.tensor((self.L - 1) / (self.out_len - 1),
                                    dtype=torch.float32, device=self.device)
                src_pos = torch.empty((self.out_len,), dtype=torch.float32, device=self.device)
                pos = torch.tensor(0.0, dtype=torch.float32, device=self.device)
                for j in range(self.out_len):
                    src_pos[j] = pos
                    pos = pos + step
                src_pos[-1] = torch.tensor(float(self.L - 1), dtype=torch.float32, device=self.device)

        left = torch.floor(src_pos).to(torch.int64).clamp(0, self.L - 1)
        right = (left + 1).clamp(0, self.L - 1)
        w = (src_pos - left.to(torch.float32)).to(torch.float32)

        self.state = StreamResamplerState(
            B=self.B, L=self.L, out_len=self.out_len,
            left=left, right=right, w=w,
            buf={}, i_in=0, j_out=0, cycles=0
        )

    @property
    def done(self) -> bool:
        return self.state.j_out >= self.state.out_len

    def step(self, x_i: Optional[torch.Tensor] = None) -> Tuple[bool, Optional[torch.Tensor], Dict[str, Any]]:
        st = self.state

        # snapshot current output requirement
        pre_j = st.j_out
        need_l = int(st.left[pre_j].item()) if pre_j < st.out_len else None
        need_r = int(st.right[pre_j].item()) if pre_j < st.out_len else None
        need_w = float(st.w[pre_j].item()) if pre_j < st.out_len else None

        # push one input sample
        pushed = False
        pushed_idx = None
        if x_i is not None:
            assert x_i.shape == (st.B,), f"x_i must be shape [B]={st.B}"
            assert x_i.dtype == torch.float32, "x_i must be FP32"
            assert st.i_in < st.L, "All inputs already pushed"
            pushed = True
            pushed_idx = st.i_in
            st.buf[st.i_in] = x_i.to(self.device).clone()
            st.i_in += 1

        # check availability after push
        have_l = (need_l in st.buf) if need_l is not None else False
        have_r = (need_r in st.buf) if need_r is not None else False

        # try produce one output
        y = None
        valid = False
        produced_j = None
        if pre_j < st.out_len and have_l and have_r:
            wl = st.w[pre_j]
            y = (1.0 - wl) * st.buf[need_l] + wl * st.buf[need_r]
            valid = True
            produced_j = pre_j
            st.j_out += 1

            # optional GC for old samples (monotonic mapping)
            min_keep = max(0, need_l - 2)
            for k in list(st.buf.keys()):
                if k < min_keep:
                    del st.buf[k]

        st.cycles += 1

        info = {
            "cycle": st.cycles,
            "pushed": pushed,
            "pushed_idx": pushed_idx,
            "pre_j_out": pre_j,
            "produced": valid,
            "produced_j": produced_j,
            "need_left": need_l,
            "need_right": need_r,
            "need_w": need_w,
            "have_left": have_l,
            "have_right": have_r,
            "buf_min": min(st.buf.keys()) if st.buf else None,
            "buf_max": max(st.buf.keys()) if st.buf else None,
            "buf_size": len(st.buf),
            "i_in_next": st.i_in,
            "j_out_next": st.j_out,
        }
        return valid, y, info

def lin_resample_1d_ref(x: torch.Tensor, out_len: int) -> torch.Tensor:
    if x.ndim == 1:
        x = x[None, :]
    B, L = x.shape
    if L == out_len:
        return x
    src_pos = torch.linspace(0, L - 1, steps=out_len, device=x.device, dtype=x.dtype)
    left = torch.floor(src_pos).long().clamp(0, L - 1)
    right = (left + 1).clamp(0, L - 1)
    w = (src_pos - left.to(x.dtype))[None, :]
    return (1 - w) * x[:, left] + w * x[:, right]

def run_streaming_full_verbose(
    x: torch.Tensor,
    out_len: int,
    mode: str = "linspace",
    max_print_cycles: int = 80,
    print_values: bool = False,
) -> Tuple[torch.Tensor, List[Dict[str, Any]]]:
    """
    Feed one input per cycle; collect outputs.
    Also returns per-cycle logs. Prints first max_print_cycles cycles.
    """
    assert x.dtype == torch.float32
    if x.ndim == 1:
        x = x[None, :]
    B, L = x.shape

    streamer = LinResample1DStreamer(L=L, out_len=out_len, B=B, device=x.device, mode=mode)

    ys = []
    logs = []
    i = 0
    while not streamer.done:
        x_i = x[:, i] if i < L else None
        if i < L:
            i += 1

        valid, y, info = streamer.step(x_i)
        if valid:
            ys.append(y)
            if print_values:
                info["y0"] = float(y[0].item()) if B == 1 else None

        logs.append(info)

        c = info["cycle"]
        if c <= max_print_cycles:
            if info["produced"]:
                msg = (f"[C{c:04d}] push={info['pushed']} idx={info['pushed_idx']} | "
                       f"OUT j={info['produced_j']} uses (l={info['need_left']}, r={info['need_right']}, w={info['need_w']:.6f}) | "
                       f"buf=[{info['buf_min']},{info['buf_max']}] size={info['buf_size']}")
                if print_values and "y0" in info:
                    msg += f" | y[0]={info['y0']:.6f}"
            else:
                msg = (f"[C{c:04d}] push={info['pushed']} idx={info['pushed_idx']} | "
                       f"STALL need (l={info['need_left']}, r={info['need_right']}, w={info['need_w']:.6f}) "
                       f"have(l={info['have_left']}, r={info['have_right']}) | "
                       f"buf=[{info['buf_min']},{info['buf_max']}] size={info['buf_size']}")
            print(msg)

    y_out = torch.stack(ys, dim=1)  # [B,out_len]
    return y_out, logs

In [7]:
torch.manual_seed(1)
x = torch.randn(1, 50, dtype=torch.float32)

y_stream, logs = run_streaming_full_verbose(x, out_len=32, max_print_cycles=50, print_values=True)
y_ref = lin_resample_1d_ref(x, 32)

print("max diff:", (y_ref - y_stream).abs().max().item())

[C0001] push=True idx=0 | STALL need (l=0, r=1, w=0.000000) have(l=True, r=False) | buf=[0,0] size=1
[C0002] push=True idx=1 | OUT j=0 uses (l=0, r=1, w=0.000000) | buf=[0,1] size=2 | y[0]=-1.525596
[C0003] push=True idx=2 | OUT j=1 uses (l=1, r=2, w=0.580645) | buf=[0,2] size=3 | y[0]=-0.694344
[C0004] push=True idx=3 | STALL need (l=3, r=4, w=0.161290) have(l=True, r=False) | buf=[0,3] size=4
[C0005] push=True idx=4 | OUT j=2 uses (l=3, r=4, w=0.161290) | buf=[1,4] size=4 | y[0]=-1.366046
[C0006] push=True idx=5 | OUT j=3 uses (l=4, r=5, w=0.741936) | buf=[2,5] size=4 | y[0]=-0.477829
[C0007] push=True idx=6 | STALL need (l=6, r=7, w=0.322581) have(l=True, r=False) | buf=[2,6] size=5
[C0008] push=True idx=7 | OUT j=4 uses (l=6, r=7, w=0.322581) | buf=[4,7] size=4 | y[0]=-1.182780
[C0009] push=True idx=8 | OUT j=5 uses (l=7, r=8, w=0.903226) | buf=[5,8] size=4 | y[0]=-0.798946
[C0010] push=True idx=9 | STALL need (l=9, r=10, w=0.483871) have(l=True, r=False) | buf=[5,9] size=5
[C0011]